# Phase 3 -- Notebook 1: Rhetorical Roles with D and C (E1, E2, E3)

| Exp | Phase-1 model | Technique | Submitted |
|---|---|---|---|
| **E1** | D encoder (reads both ways) | **LoRMA** (compulsory) | yes |
| **E2** | D encoder | LoRA (same rank, same layers as E1) | yes |
| **E3** | C (prompt read both ways) | IA3 | yes |
| E1_noctx | D encoder | LoRMA, **no document-context layer** | no -- dev-only ablation: does context help? |

Every experiment uses the same pipeline: sentence -> frozen Phase-1 backbone + adapter -> mean-pooled
vector (+ relative position + domain flag) -> 2-layer BiLSTM over the document's sentences -> 13 role
logits. Class-weighted loss for rare roles; one bias per class fitted on dev for macro-F1 (kept only if a
2-fold check on dev says it helps). E1 and E2 also save their logits on train/dev/test: notebook 2 uses
the better one as the **distillation teacher**.

**How to run**
1. Inputs: `legallm-tokenizers`, `legallm-checkpoints-d`, `legallm-checkpoints-c`.
   (For D, attach the dataset version holding the checkpoint you want -- see `D_CKPT_NAME` below.)
2. Secret on: `HF`. Accelerator **GPU T4 x2**, internet on.
3. **Save Version -> Save & Run All.** The two GPUs run two experiments at a time
   (E1 | E2, then E3 | E1_noctx). **Time: ~80-90 min.**
4. Outputs: `p3_nb1/E1/submission.zip`, `p3_nb1/E2/submission.zip`, `p3_nb1/E3/submission.zip`,
   plus `runs.csv`, per-experiment `metrics.json`, `dev_confusion.csv`, `logits.pt`, and `rr_data.pt`.
   **Attach this notebook's output to notebook 2.**


## 1. Paths and checkpoints

In [1]:
import csv
import json
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

import torch

N_GPUS = torch.cuda.device_count()
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "p3_nb1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

D_CKPT_NAME = None      # None = newest D_step*.pt attached; or e.g. "D_step10585_tokens650342400.pt"


def find(pattern, label):
    hits = sorted(KAGGLE_INPUT.rglob(pattern)) if KAGGLE_INPUT.exists() else []
    assert hits, f"Nothing matching {pattern} under /kaggle/input -- attach {label}."
    return hits


def step_of(p):
    return int(Path(p).stem.split("_step")[1].split("_")[0])


TOKENIZER_PATH = find("t2_unigram.model", "legallm-tokenizers")[0]
d_all = find("D_step*.pt", "legallm-checkpoints-d")
D_CKPT = next(p for p in d_all if p.name == D_CKPT_NAME) if D_CKPT_NAME else max(d_all, key=step_of)
C_CKPT = max(find("C_step*.pt", "legallm-checkpoints-c"), key=step_of)
print("Tokenizer:", TOKENIZER_PATH)
print("D checkpoint:", D_CKPT.name, "| all found:", [p.name for p in d_all])
print("C checkpoint:", C_CKPT.name)


PyTorch 2.11.0+cu128 | GPUs: ['Tesla T4', 'Tesla T4']
Tokenizer: /kaggle/input/datasets/jbhusal/legallm-tokenizers/t2_unigram.model
D checkpoint: D_step10585_tokens650342400.pt | all found: ['D_step10585_tokens650342400.pt']
C checkpoint: C_step13683_tokens448364544.pt


In [2]:
# FIX: Kaggle unzipped the uploaded checkpoints into folders -> re-pack them into .pt files
import zipfile
from pathlib import Path


def repack(p):
    p = Path(p)
    if p.is_file():
        return p
    root = next(p.rglob("data.pkl")).parent
    name = p.name if p.name.endswith(".pt") else p.name + ".pt"
    dest = Path("/kaggle/working/ckpts") / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_STORED) as zf:
        for f in sorted(root.rglob("*")):
            if f.is_file():
                zf.write(f, arcname="archive/" + f.relative_to(root).as_posix())
    print("re-packed", p, "->", dest)
    return dest


D_CKPT = repack(D_CKPT)
C_CKPT = repack(C_CKPT)
print("D:", D_CKPT, "| C:", C_CKPT)

re-packed /kaggle/input/datasets/jbhusal/phase1-checkpoint-d/D_step10585_tokens650342400.pt -> /kaggle/working/ckpts/D_step10585_tokens650342400.pt
re-packed /kaggle/input/datasets/jbhusal/phase1-checkpoint-c/C_step13683_tokens448364544.pt -> /kaggle/working/ckpts/C_step13683_tokens448364544.pt
D: /kaggle/working/ckpts/D_step10585_tokens650342400.pt | C: /kaggle/working/ckpts/C_step13683_tokens448364544.pt


## 2. Code files (written to /kaggle/working)

In [3]:
%%writefile /kaggle/working/legal_d.py
"""legal_d.py -- Model D: encoder-decoder legal language model + its trainer.

Model D (T5 / RedLLM-style encoder-decoder, tokenizer T2):
  * Encoder: bidirectional self-attention over the prompt.
  * Decoder: causal self-attention over the continuation + cross-attention
    to the encoder output.
  * Shared (tied) embedding for encoder input, decoder input and output head.
  * RMSNorm (pre-norm), RoPE in self-attention, QK-norm, GQA, SwiGLU, no biases.

Trainer: one process per GPU (DDP via torch.distributed.run), Muon optimizer
for the hidden weight matrices + AdamW for embeddings/norms, fp16 autocast
with GradScaler (T4 has no bf16), time-based warmup-stable-decay LR
schedule (so the LR always finishes its decay when the time budget ends),
flat checkpoints with resume, periodic push to a Kaggle dataset.

Run training:   python -m torch.distributed.run --standalone --nproc_per_node=2 legal_d.py d_config.json
Import helpers: import legal_d  (model, generation, scoring)
"""
import contextlib
import datetime
import itertools
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

ARCH_ID = "D"

MODEL_D_CONFIG = dict(
    vocab_size=32_000,        # T2 (SentencePiece Unigram), same as C
    d_model=768,
    n_enc_layer=12,
    n_dec_layer=12,
    n_head=12,
    n_kv_head=4,              # GQA 3:1 (self- and cross-attention)
    d_head=64,
    d_ff=2048,                # SwiGLU (same FFN parameter count as C's 4x GELU)
    max_enc_len=1024,
    max_dec_len=1024,
    qk_norm=True,
    mlp="swiglu",
    rope_theta=10_000.0,
    tokenizer_id="T2",
    arch="encoder-decoder",
    tied_embeddings=True,
)

# ---------------------------------------------------------------------------
# Tokenizer (T2)
# ---------------------------------------------------------------------------
sp = None
EOS_ID = None
DEC_START_ID = None
BANNED_IN_OUTPUT = []


def load_tokenizer(path):
    global sp, EOS_ID, DEC_START_ID, BANNED_IN_OUTPUT
    import sentencepiece as spm
    sp = spm.SentencePieceProcessor()
    sp.load(str(path))
    EOS_ID = sp.eos_id()
    assert EOS_ID >= 0, "T2 tokenizer has no <eos>"
    DEC_START_ID = sp.bos_id() if sp.bos_id() >= 0 else EOS_ID      # decoder start token
    BANNED_IN_OUTPUT = sorted({sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()} - {-1})
    return sp


def encode(text):
    return sp.encode(text, out_type=int)


def decode(ids):
    return sp.decode(ids)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(x.dtype)


def rope_cache(n, d_head, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(n).float(), inv_freq)
    return freqs.cos(), freqs.sin()                      # [n, d_head/2]


def apply_rope(x, cos, sin):
    """x [B, H, T, d]; cos/sin [T, d/2]. Rotate-half form, computed in fp32."""
    cos, sin = cos[None, None].float(), sin[None, None].float()
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half].float(), x[..., half:].float()
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


def expand_kv(t, n_rep):
    if n_rep == 1:
        return t
    B, H, T, D = t.shape
    return t[:, :, None].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)


class KVCache:
    """Decoder self-attention cache (allocated on first use)."""

    def __init__(self, max_len):
        self.max_len, self.pos, self.k, self.v = max_len, 0, None, None

    def update(self, k, v):
        if self.k is None:
            B, H, _, D = k.shape
            self.k = torch.zeros(B, H, self.max_len, D, device=k.device, dtype=k.dtype)
            self.v = torch.zeros_like(self.k)
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class SelfAttention(nn.Module):
    def __init__(self, cfg, causal):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.causal, self.n_head, self.n_kv_head, self.d_head = causal, h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, key_mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        first_chunk = cache is None or cache.pos == 0
        if cache is not None:
            assert T == 1 or cache.pos == 0, "cached decoding feeds one token at a time"
            k, v = cache.update(k, v)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(k, rep), expand_kv(v, rep)
        if key_mask is not None:                                  # [B, Tk] True = real token
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask[:, None, None, :])
        else:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal and T > 1 and first_chunk)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class CrossAttention(nn.Module):
    """Decoder queries attend to the encoder output (no RoPE: T5-style)."""

    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=False)
        self.k_proj = nn.Linear(d, kv * dh, bias=False)
        self.v_proj = nn.Linear(d, kv * dh, bias=False)
        self.o_proj = nn.Linear(h * dh, d, bias=False)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def project_kv(self, enc_out):
        B, S, _ = enc_out.shape
        k = self.k_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_out).view(B, S, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.k_norm is not None:
            k = self.k_norm(k)
        return k, v

    def forward(self, x, enc_kv, enc_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q = self.q_norm(q)
        rep = self.n_head // self.n_kv_head
        k, v = expand_kv(enc_kv[0], rep), expand_kv(enc_kv[1], rep)
        mask = enc_mask[:, None, None, :] if enc_mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg["d_model"], cfg["d_ff"]
        self.gate_proj = nn.Linear(d, f, bias=False)
        self.up_proj = nn.Linear(d, f, bias=False)
        self.down_proj = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = SelfAttention(cfg, causal=False)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, key_mask=None):
        x = x + self.attn(self.attn_norm(x), cos, sin, key_mask)
        return x + self.mlp(self.mlp_norm(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_norm = RMSNorm(cfg["d_model"])
        self.self_attn = SelfAttention(cfg, causal=True)
        self.cross_norm = RMSNorm(cfg["d_model"])
        self.cross_attn = CrossAttention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = SwiGLU(cfg)

    def forward(self, x, cos, sin, enc_out=None, enc_mask=None, enc_kv=None, cache=None):
        if enc_kv is None:
            enc_kv = self.cross_attn.project_kv(enc_out)
        x = x + self.self_attn(self.self_norm(x), cos, sin, cache=cache)
        x = x + self.cross_attn(self.cross_norm(x), enc_kv, enc_mask)
        return x + self.mlp(self.mlp_norm(x))


def _ce_chunk(h, w, t):
    logits = F.linear(h, w).float()
    lse = torch.logsumexp(logits, dim=-1)
    valid = (t != -100).float()
    nll = (lse - logits.gather(-1, t.clamp(min=0)[:, None]).squeeze(-1)) * valid
    return torch.stack([nll.sum(), (lse.pow(2) * valid).sum()])


def chunked_loss(h, w, targets, chunk=4096):
    """[1, 3] = [sum of token NLL, number of scored tokens, sum of logsumexp^2].
    Logits are built chunk by chunk (and recomputed in backward), so the full
    [tokens x 32k] logit tensor is never stored."""
    h = h.reshape(-1, h.shape[-1])
    t = targets.reshape(-1)
    parts = []
    for s in range(0, h.shape[0], chunk):
        hs, ts = h[s:s + chunk], t[s:s + chunk]
        if torch.is_grad_enabled() and h.requires_grad:
            parts.append(grad_checkpoint(_ce_chunk, hs, w, ts, use_reentrant=False))
        else:
            parts.append(_ce_chunk(hs, w, ts))
    tot = torch.stack(parts).sum(0)
    count = (t != -100).sum().to(tot.dtype)
    return torch.stack([tot[0], count, tot[1]]).unsqueeze(0)


class EncoderDecoderLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d, V = config["d_model"], config["vocab_size"]
        self.embed = nn.Embedding(V, d)
        self.enc_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config["n_enc_layer"])])
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config["n_dec_layer"])])
        self.dec_norm = RMSNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.embed.weight                  # tied
        cos, sin = rope_cache(max(config["max_enc_len"], config["max_dec_len"]), config["d_head"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.grad_ckpt = False
        self.amp = True                                        # fp16 autocast on GPU (fp32 weights)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
        for name, p in self.named_parameters():                 # scaled residual-output init
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                n_res = 2 * config["n_enc_layer"] if name.startswith("enc_") else 3 * config["n_dec_layer"]
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(n_res))

    def _run(self, blk, *args, **kwargs):
        if self.grad_ckpt and self.training and torch.is_grad_enabled():
            return grad_checkpoint(blk, *args, use_reentrant=False, **kwargs)
        return blk(*args, **kwargs)

    def encode(self, input_ids, attention_mask=None):
        """Contract method: encoder hidden states [B, T, d]. attention_mask
        (1 = real token) marks padding; positions start at 0 in every row
        (pad on the right)."""
        T = input_ids.shape[1]
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = attention_mask.bool() if attention_mask is not None else None
        for blk in self.enc_blocks:
            x = self._run(blk, x, cos, sin, key_mask)
        return self.enc_norm(x)

    def decode_hidden(self, dec_ids, enc_out, enc_mask=None):
        T = dec_ids.shape[1]
        x = self.embed(dec_ids)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        m = enc_mask.bool() if enc_mask is not None else None
        for blk in self.dec_blocks:
            x = self._run(blk, x, cos, sin, enc_out=enc_out, enc_mask=m)
        return self.dec_norm(x)

    def forward(self, enc_ids, dec_ids, targets=None, enc_mask=None):
        """No targets -> logits [B, Q, V]. With targets -> [1, 3] sums
        (NLL, count, lse^2); targets = -100 are ignored."""
        use_amp = enc_ids.is_cuda and self.embed.weight.dtype == torch.float32 and self.amp
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            enc_out = self.encode(enc_ids, enc_mask)
            h = self.decode_hidden(dec_ids, enc_out, enc_mask)
            if targets is None:
                return self.head(h)
            return chunked_loss(h, self.head.weight, targets)

    def decoder_step(self, tok, pos, caches, cross_kvs, enc_mask):
        x = self.embed(tok)
        cos, sin = self.rope_cos[pos:pos + tok.shape[1]], self.rope_sin[pos:pos + tok.shape[1]]
        for blk, cache, kv in zip(self.dec_blocks, caches, cross_kvs):
            x = blk(x, cos, sin, enc_mask=enc_mask, enc_kv=kv, cache=cache)
        return self.head(self.dec_norm(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (one prompt, greedy): returns prompt + continuation ids."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


def build_model(config):
    """Contract function."""
    return EncoderDecoderLM(config)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# ---------------------------------------------------------------------------
# Generation
# ---------------------------------------------------------------------------
def _sample_top_p(logits, temperature, top_p, generator):
    probs = torch.softmax(logits / temperature, dim=-1)
    if top_p < 1.0:
        sorted_p, idx = probs.sort(dim=-1, descending=True)
        drop = (sorted_p.cumsum(-1) - sorted_p) > top_p
        sorted_p = sorted_p.masked_fill(drop, 0.0)
        probs = torch.zeros_like(probs).scatter_(-1, idx, sorted_p)
    return torch.multinomial(probs, 1, generator=generator).squeeze(-1)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_enc_len=768, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None, sample=False,
                   temperature=1.0, top_p=1.0, generator=None, start_id=None):
    """Continuations for several prompts at once. Encoder input is right-padded
    (masked); every decoder row starts at the same position, so no decoder
    padding is needed. Greedy by default; `sample=True` gives temperature /
    top-p sampling (used for MBR candidates). Optional repetition penalty over
    generated tokens, n-gram blocking, and early stop once every row has more
    than `target_words` words."""
    model.eval()
    device = model.embed.weight.device
    cfg = model.config
    start_id = DEC_START_ID if start_id is None else start_id
    max_enc_len = min(max_enc_len or cfg["max_enc_len"], cfg["max_enc_len"])
    prompts = [(p[-max_enc_len:] if p else [EOS_ID if EOS_ID is not None else 0]) for p in prompt_id_lists]
    B, P = len(prompts), max(len(p) for p in prompts)
    enc_ids = torch.zeros(B, P, dtype=torch.long, device=device)
    enc_mask = torch.zeros(B, P, dtype=torch.bool, device=device)
    for b, p in enumerate(prompts):
        enc_ids[b, :len(p)] = torch.tensor(p, device=device)
        enc_mask[b, :len(p)] = True
    use_amp = device.type == "cuda" and model.embed.weight.dtype == torch.float32 and getattr(model, "amp", True)
    amp = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
    with amp:
        enc_out = model.encode(enc_ids, enc_mask)
        cross_kvs = [blk.cross_attn.project_kv(enc_out) for blk in model.dec_blocks]
        caches = [KVCache(max_new_tokens + 1) for _ in model.dec_blocks]
        tok = torch.full((B, 1), start_id, dtype=torch.long, device=device)
        logits = model.decoder_step(tok, 0, caches, cross_kvs, enc_mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = _sample_top_p(logits, temperature, top_p, generator) if sample else logits.argmax(dim=-1)
        for b, tk in enumerate(nxt.tolist()):
            generated[b].append(tk)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(len(decode(g).split()) > target_words for g in generated):
            break
        with amp:
            logits = model.decoder_step(nxt[:, None], t + 1, caches, cross_kvs, enc_mask)[:, -1, :].float()
    return generated


# ---------------------------------------------------------------------------
# Scoring helpers (mirror evaluate_phase1.py; BERTScore uses roberta-large)
# ---------------------------------------------------------------------------
def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return bin(state).count("1")


def rouge_l_f1(p_text, r_text):
    pt, rt = p_text.split(), r_text.split()
    if not pt or not rt:
        return 0.0
    n = lcs_length(pt, rt)
    if not n:
        return 0.0
    pr, rc = n / len(pt), n / len(rt)
    return 2 * pr * rc / (pr + rc)


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)) if len(tokens) >= n else Counter()


def bleu4(preds, refs):
    clipped, totals, pl, rl = [0] * 4, [0] * 4, 0, 0
    for p, r in zip(preds, refs):
        pl += len(p)
        rl += len(r)
        for n in range(1, 5):
            pn, rn = _ngrams(p, n), _ngrams(r, n)
            totals[n - 1] += sum(pn.values())
            for g, c in pn.items():
                clipped[n - 1] += min(c, rn.get(g, 0))
    prec = [clipped[i] / totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x <= 0 for x in prec) or pl == 0:
        return 0.0
    return (1.0 if pl > rl else math.exp(1.0 - rl / pl)) * math.exp(sum(math.log(x) for x in prec) / 4)


def score_predictions(pred, truth, ids):
    try:
        import bert_score
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert-score"], check=True)
        import bert_score
    hyps = [pred[i]["text"].strip() for i in ids]
    refs = [truth[i]["text"].strip() for i in ids]
    _, _, f1 = bert_score.score(hyps, refs, lang="en", batch_size=8, verbose=False,
                                device="cuda" if torch.cuda.is_available() else "cpu")
    return {"bertscore": float(f1.mean()),
            "rouge_l": sum(rouge_l_f1(h, r) for h, r in zip(hyps, refs)) / len(refs),
            "bleu4": bleu4([h.split() for h in hyps], [r.split() for r in refs])}


def to_words(toks, n_words=128):
    words = decode(toks).split()[:n_words]
    return " ".join(words) if words else "."


def predict(model, prompt_dict, ids, batch_size=16, desc="predict", **settings):
    """Greedy/sampled continuations -> {id: {"text": 128 words}}. Prompts are
    sorted by length so each batch has similar lengths."""
    from tqdm.auto import tqdm
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_size), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_size]
        for i, toks in zip(chunk, generate_batch(model, [enc[i] for i in chunk], **settings)):
            out[i] = {"text": to_words(toks)}
    return {i: out[i] for i in ids}


def mbr_pick(candidates):
    """Minimum-Bayes-risk choice: the candidate with the highest average
    ROUGE-L F1 against the other candidates (a consensus pick; no outside
    model involved)."""
    if len(candidates) <= 2:
        return candidates[0]
    best, best_u = 0, -1.0
    for i, c in enumerate(candidates):
        u = sum(rouge_l_f1(c, o) for j, o in enumerate(candidates) if j != i) / (len(candidates) - 1)
        if u > best_u:
            best, best_u = i, u
    return candidates[best]


def predict_mbr(model, prompt_dict, ids, n_samples=7, batch_prompts=8, greedy_settings=None,
                sample_settings=None, seed=1234, desc="mbr"):
    """For each prompt: 1 greedy candidate + n_samples sampled candidates ->
    MBR pick. Candidates of `batch_prompts` prompts are generated together."""
    from tqdm.auto import tqdm
    device = model.embed.weight.device
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    enc = {i: encode(prompt_dict[i]["text"]) for i in ids}
    order = sorted(ids, key=lambda i: len(enc[i]))
    out = {}
    for s in tqdm(range(0, len(order), batch_prompts), desc=desc, dynamic_ncols=True, mininterval=30):
        chunk = order[s:s + batch_prompts]
        greedy = generate_batch(model, [enc[i] for i in chunk], **greedy_settings)
        sampled = generate_batch(model, [enc[i] for i in chunk for _ in range(n_samples)], sample=True,
                                 generator=gen, **sample_settings)
        for k, i in enumerate(chunk):
            cands = [to_words(greedy[k])] + [to_words(t) for t in sampled[k * n_samples:(k + 1) * n_samples]]
            out[i] = {"text": mbr_pick(cands)}
    return {i: out[i] for i in ids}


@torch.no_grad()
def dev_loss_bpb(model, dev_prompts, dev_truth, ids, max_enc_len=768, max_dec_len=512):
    """Continuation loss per token and bits-per-byte (tokenizer-independent)."""
    model.eval()
    device = model.embed.weight.device
    nll, n_tok, n_bytes = 0.0, 0, 0
    for i in ids:
        cont_text = " " + dev_truth[i]["text"].strip()
        cont = encode(cont_text)[:max_dec_len]
        ctx = encode(dev_prompts[i]["text"])[-max_enc_len:]
        if not cont or not ctx:
            continue
        enc_ids = torch.tensor([ctx], device=device)
        dec_in = torch.tensor([[DEC_START_ID] + cont[:-1]], device=device)
        tgt = torch.tensor([cont], device=device)
        out = model(enc_ids, dec_in, targets=tgt)[0]
        nll += out[0].item()
        n_tok += len(cont)
        n_bytes += len(cont_text.encode("utf-8"))
    return nll / max(1, n_tok), nll / (math.log(2) * max(1, n_bytes))


def self_test(device):
    """Checks: cached decoding == full forward; decoder is causal; encoder is
    two-way; encoder padding changes nothing; loss sums are correct; batched
    generation == one-at-a-time; Muon orthogonalises."""
    torch.manual_seed(0)
    tiny = dict(MODEL_D_CONFIG, vocab_size=101, d_model=64, n_enc_layer=2, n_dec_layer=2, n_head=4,
                n_kv_head=2, d_head=16, d_ff=96, max_enc_len=64, max_dec_len=64)
    m = build_model(tiny).to(device).eval()
    m.amp = False                                          # exactness checks in fp32 (GPU kernels still used)
    enc = torch.randint(4, 101, (2, 20), device=device)
    dec = torch.randint(4, 101, (2, 12), device=device)
    with torch.no_grad():
        full = m(enc, dec).float()
        enc_out = m.encode(enc)
        kvs = [blk.cross_attn.project_kv(enc_out) for blk in m.dec_blocks]
        caches = [KVCache(12) for _ in m.dec_blocks]
        steps = [m.decoder_step(dec[:, t:t + 1], t, caches, kvs, None) for t in range(12)]
        diff = (full - torch.cat(steps, 1).float()).abs().max().item()
        dec2 = dec.clone()
        dec2[:, 8] = (dec[:, 8] + 1) % 97 + 4
        leak = (m(enc, dec2)[:, :8] - full[:, :8]).abs().max().item()
        enc2 = enc.clone()
        enc2[:, 15] = (enc[:, 15] + 1) % 97 + 4
        two_way = (m.encode(enc2)[:, 2] - enc_out[:, 2]).abs().max().item()
        padded = torch.cat([enc, torch.zeros(2, 5, dtype=torch.long, device=device)], 1)
        mask = torch.cat([torch.ones(2, 20), torch.zeros(2, 5)], 1).to(device)
        pad_diff = (m(padded, dec, enc_mask=mask) - full).abs().max().item()
        tgt = torch.randint(4, 101, (2, 12), device=device)
        tgt[0, 3] = -100
        s = m(enc, dec, targets=tgt)[0]
        ref = F.cross_entropy(full.reshape(-1, 101), tgt.reshape(-1), ignore_index=-100, reduction="sum")
    assert diff < 1e-3, f"cached decoding != full forward ({diff})"
    assert leak < 1e-5, f"decoder sees future tokens ({leak})"
    assert two_way > 1e-5, "encoder is not bidirectional"
    assert pad_diff < 1e-4, f"encoder padding changes the output ({pad_diff})"
    assert abs(s[0].item() - ref.item()) < 1e-3 and int(s[1].item()) == 23, "loss sums wrong"
    pa, pb = enc[0].tolist(), enc[1, :9].tolist()
    both = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1)
    alone = [generate_batch(m, [pa], max_new_tokens=8, start_id=1)[0],
             generate_batch(m, [pb], max_new_tokens=8, start_id=1)[0]]
    assert both == alone, "batched generation differs from one-at-a-time"
    g = torch.Generator(device=device)
    g.manual_seed(0)
    smp = generate_batch(m, [pa, pb], max_new_tokens=8, start_id=1, sample=True, temperature=0.8, top_p=0.9,
                         generator=g)
    assert len(smp) == 2 and all(len(x) == 8 for x in smp)
    X = newton_schulz5(torch.randn(48, 96, device=device)).float()
    sv = torch.linalg.svdvals(X)
    assert 0.3 < sv.min().item() and sv.max().item() < 1.5, f"Muon orthogonalisation off: {sv.min():.2f}..{sv.max():.2f}"
    print(f"Self-test passed: cached==full {diff:.1e}, causal decoder {leak:.1e}, two-way encoder, "
          f"padding-safe {pad_diff:.1e}, loss sums ok, batched==single, sampling ok, "
          f"Muon singular values {sv.min():.2f}..{sv.max():.2f}.")


# ---------------------------------------------------------------------------
# Muon optimizer (Jordan et al. 2024; update scale from Liu et al. 2025,
# "Muon is Scalable for LLM Training": 0.2 * sqrt(max(rows, cols)) so the
# learning rate and weight decay work on AdamW's scale).
# ---------------------------------------------------------------------------
def newton_schulz5(G, steps=5, eps=1e-7):
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.float()
    X = X / (X.norm() + eps)                              # normalise in fp32 (tiny grads survive)
    if G.is_cuda:
        X = X.half()                                       # T4: fast fp16 matmuls, values stay <= 1
    transposed = X.shape[0] > X.shape[1]
    if transposed:
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        Bm = b * A + c * (A @ A)
        X = a * X + Bm @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.95, nesterov=True, weight_decay=0.1, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov,
                                      weight_decay=weight_decay, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, mom, wd = group["lr"], group["momentum"], group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad.float()
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf = state["momentum_buffer"]
                buf.mul_(mom).add_(g)
                g = g.add(buf, alpha=mom) if group["nesterov"] else buf
                u = newton_schulz5(g, steps=group["ns_steps"]).float()
                p.mul_(1.0 - lr * wd)
                p.add_(u, alpha=-lr * 0.2 * math.sqrt(max(p.shape[0], p.shape[1])))
        return None


def build_optimizers(model, kind, lr, weight_decay):
    muon_params, adam_decay, adam_no_decay = [], [], []
    for name, p in model.named_parameters():
        if p.dim() == 2 and (name.startswith("enc_blocks") or name.startswith("dec_blocks")) and kind == "muon":
            muon_params.append(p)
        elif p.dim() >= 2:
            adam_decay.append(p)                           # embedding (tied output head); all matrices for AdamW
        else:
            adam_no_decay.append(p)                        # norm weights
    adam = torch.optim.AdamW([{"params": adam_decay, "weight_decay": weight_decay},
                              {"params": adam_no_decay, "weight_decay": 0.0}], lr=lr, betas=(0.9, 0.95), eps=1e-8)
    return ([Muon(muon_params, lr=lr, weight_decay=weight_decay)] if muon_params else []) + [adam]


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def lr_factor(step, progress, warmup_steps, cooldown_frac, floor):
    """Warmup-stable-decay on *time*: linear warmup (steps), constant, then a
    1-sqrt cooldown over the last `cooldown_frac` of the training time."""
    w = min(1.0, (step + 1) / max(1, warmup_steps))
    start = 1.0 - cooldown_frac
    if progress < start:
        return w
    r = min(1.0, (progress - start) / max(1e-9, cooldown_frac))
    return w * (floor + (1.0 - floor) * (1.0 - math.sqrt(r)))


def loss_from(out, z_loss_coef):
    s = out.sum(0)
    n = s[1].clamp(min=1)
    return s[0] / n, (s[0] + z_loss_coef * s[2]) / n


# ---------------------------------------------------------------------------
# Data: stream of 1024-token windows (round-robin over the cleaned files,
# documents separated by <eos>), byte positions saved for resume. Each GPU
# (rank) starts at a different offset of every file.
# ---------------------------------------------------------------------------
class WindowStream:
    def __init__(self, file_paths, window, rank=0, world=1, start_positions=None):
        self.files = [Path(p) for p in file_paths if Path(p).stat().st_size > 0]
        self.window = window
        self.positions, self.unaligned = {}, set()
        for p in self.files:
            if start_positions and p.name in start_positions:
                self.positions[p.name] = int(start_positions[p.name])
            else:
                self.positions[p.name] = (p.stat().st_size * rank) // world
                if self.positions[p.name] > 0:
                    self.unaligned.add(p.name)

    def __iter__(self):
        buffer = []
        readers = [self._read_lines(p) for p in self.files]
        for reader in itertools.cycle(readers):
            ids = encode(next(reader))
            if not ids:
                continue
            buffer.extend(ids)
            buffer.append(EOS_ID)
            while len(buffer) >= self.window:
                yield buffer[:self.window]
                buffer = buffer[self.window:]

    def _read_lines(self, path):
        name, size = path.name, path.stat().st_size
        while True:
            start = self.positions[name] if self.positions[name] < size else 0
            with open(path, "rb") as f:
                f.seek(start)
                pos = start
                if name in self.unaligned:                  # started mid-line: skip to the next line
                    pos += len(f.readline())
                    self.unaligned.discard(name)
                while True:
                    raw = f.readline()
                    if not raw:
                        break
                    pos += len(raw)
                    self.positions[name] = pos
                    line = raw.decode("utf-8", errors="ignore").strip()
                    if line:
                        yield line
            self.positions[name] = 0


def make_example_batch(windows, P, Q, device):
    """windows: list of 1024-token lists -> encoder ids [B,P], decoder input
    [B,Q] (start token + shifted continuation), targets [B,Q]."""
    w = torch.tensor(windows, dtype=torch.long)
    enc = w[:, :P]
    tgt = w[:, P:P + Q]
    dec_in = torch.cat([torch.full((w.shape[0], 1), DEC_START_ID, dtype=torch.long), tgt[:, :-1]], dim=1)
    return enc.to(device, non_blocking=True), dec_in.to(device, non_blocking=True), tgt.to(device, non_blocking=True)


def choose_shape(shapes, seed, step, micro):
    """Same (prompt, continuation) split on every GPU for a given micro-step."""
    rnd = random.Random(seed * 1_000_003 + step * 97 + micro)
    return rnd.choices([(s[0], s[1]) for s in shapes], weights=[s[2] for s in shapes])[0]


# ---------------------------------------------------------------------------
# Checkpoints and Kaggle push
# ---------------------------------------------------------------------------
def step_of(path):
    return int(Path(path).stem.split("_step")[1].split("_tokens")[0])


def find_latest_checkpoint(prefix, search_dirs, extract_to):
    cands = []
    for d in search_dirs:
        d = Path(d)
        if d.exists():
            cands += [p for p in d.rglob(f"{prefix}_step*.pt")]
    best_zip = None
    for d in search_dirs:
        d = Path(d)
        if not d.exists():
            continue
        for zpath in d.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zpath) as zf:
                    for name in zf.namelist():
                        base = Path(name).name
                        if base.startswith(f"{prefix}_step") and base.endswith(".pt"):
                            if best_zip is None or step_of(base) > best_zip[0]:
                                best_zip = (step_of(base), zpath, name)
            except zipfile.BadZipFile:
                continue
    best_file = max(cands, key=step_of) if cands else None
    if best_zip is not None and (best_file is None or best_zip[0] > step_of(best_file)):
        _, zpath, name = best_zip
        dest = Path(extract_to) / Path(name).name
        with zipfile.ZipFile(zpath) as zf, zf.open(name) as src, open(dest, "wb") as dst:
            shutil.copyfileobj(src, dst)
        return dest
    return best_file


def prune_old_checkpoints(ckpt_dir, prefix, keep=2):
    ckpts = sorted(Path(ckpt_dir).glob(f"{prefix}_step*.pt"), key=step_of)
    for p in ckpts[:-keep] if len(ckpts) > keep else []:
        p.unlink()


def push_checkpoint_dataset(output_dir, dataset_id, message):
    kdir = Path.home() / ".kaggle"
    if not ((kdir / "access_token").exists() or (kdir / "kaggle.json").exists() or os.environ.get("KAGGLE_API_TOKEN")):
        print("Skipping checkpoint push: no Kaggle API token.", flush=True)
        return False
    slug = dataset_id.split("/")[1]
    with open(Path(output_dir) / "dataset-metadata.json", "w") as f:
        json.dump({"title": slug, "id": dataset_id, "licenses": [{"name": "CC0-1.0"}]}, f)
    try:
        exists = subprocess.run(["kaggle", "datasets", "status", dataset_id], capture_output=True, text=True,
                                timeout=300).returncode == 0
        cmd = (["kaggle", "datasets", "version", "-p", str(output_dir), "-m", message, "-r", "zip"] if exists
               else ["kaggle", "datasets", "create", "-p", str(output_dir), "-r", "zip"])
        t0 = time.time()
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    except (OSError, subprocess.TimeoutExpired) as exc:
        print(f"Checkpoint push failed: {exc}", flush=True)
        return False
    if result.returncode != 0:
        print(f"Checkpoint push failed: {(result.stderr or result.stdout).strip()[-500:]}", flush=True)
        return False
    print(f"Pushed checkpoint dataset in {time.time() - t0:.0f}s ({message}).", flush=True)
    return True


def load_for_inference(path, device, half=False):
    """fp32 weights + fp16 autocast on GPU (same numerics as training)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = build_model(ckpt["config"])
    model.load_state_dict(ckpt["model_state"])
    meta = {k: ckpt.get(k) for k in ("step", "tokens_seen", "config", "run_id")}
    del ckpt
    model = model.to(device).eval()
    if half and device.type == "cuda":
        model = model.half()
        model.rope_cos, model.rope_sin = model.rope_cos.float(), model.rope_sin.float()
    return model, meta


# ---------------------------------------------------------------------------
# Trainer (one process per GPU)
# ---------------------------------------------------------------------------
def _probe_micro_batch(model, cfg, device, P, Q):
    """Largest micro-batch (1024-token examples) that fits the memory budget
    on this GPU, with the real optimizers; weights are restored afterwards."""
    saved = {k: v.detach().clone().cpu() for k, v in model.state_dict().items()}
    results = {}
    for bs in cfg["candidate_micro_bs"]:
        optims = build_optimizers(model, cfg["optimizer"], 1e-5, 0.0)
        scaler = make_grad_scaler(True)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        try:
            windows = [[random.randrange(4, 1000) for _ in range(P + Q)] for _ in range(bs)]
            enc, dec_in, tgt = make_example_batch(windows, P, Q, device)
            times = []
            for _ in range(3):
                torch.cuda.synchronize(device)
                t0 = time.time()
                for o in optims:
                    o.zero_grad(set_to_none=True)
                _, total = loss_from(model(enc, dec_in, targets=tgt), 1e-4)
                scaler.scale(total).backward()
                for o in optims:
                    scaler.step(o)
                scaler.update()
                torch.cuda.synchronize(device)
                times.append(time.time() - t0)
            peak = torch.cuda.max_memory_allocated(device) / 1e9
            ok = peak <= cfg["memory_budget_gb"]
            results[bs] = (peak, min(times[1:]), ok)
            if not ok:
                break
        except Exception as exc:                           # noqa: BLE001
            if "out of memory" not in str(exc).lower():
                raise
            results[bs] = (None, None, False)
            break
        finally:
            for o in optims:
                o.zero_grad(set_to_none=True)
            del optims
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    model.load_state_dict(saved)
    return results


def train_main(config_path):
    cfg = json.load(open(config_path))
    distributed = int(os.environ.get("WORLD_SIZE", "1")) > 1
    cuda = torch.cuda.is_available()
    if distributed:
        dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=90))
        rank, world, local_rank = dist.get_rank(), dist.get_world_size(), int(os.environ.get("LOCAL_RANK", 0))
    else:
        rank, world, local_rank = 0, 1, 0
    device = torch.device(f"cuda:{local_rank}") if cuda else torch.device("cpu")
    if cuda:
        torch.cuda.set_device(device)
        torch.backends.cuda.matmul.allow_tf32 = True
    main = rank == 0

    def log(*a):
        if main:
            print(*a, flush=True)

    load_tokenizer(cfg["tokenizer_path"])
    model_cfg = cfg["model"]
    prefix = cfg["ckpt_prefix"]
    ckpt_dir = Path(cfg["checkpoint_dir"])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    session_start, session_deadline = cfg["session_start_epoch"], cfg["session_start_epoch"] + 60 * cfg["time_budget_min"]
    total_seconds = 3600 * cfg["total_train_hours"]
    log(f"[D] world={world} device={device} | budget this session {cfg['time_budget_min']} min, "
        f"whole run {cfg['total_train_hours']} h | optimizer={cfg['optimizer']}")

    torch.manual_seed(cfg["seed"])
    random.seed(cfg["seed"] + rank)
    model = build_model(model_cfg).to(device)
    n_params = count_params(model)
    log(f"[D] parameters: {n_params / 1e6:.1f}M")

    # ---- resume ----------------------------------------------------------
    step, tokens_seen, seconds_before, rank_positions, ckpt = 0, 0, 0.0, None, None
    phase = cfg.get("phase_id", "main")        # a new phase id = continue training with a fresh LR schedule
    phase_start_step = 0
    latest = find_latest_checkpoint(prefix, [ckpt_dir] + cfg.get("resume_search_dirs", []), ckpt_dir) if main else None
    if distributed:
        box = [str(latest) if latest else ""]
        dist.broadcast_object_list(box, src=0)
        latest = Path(box[0]) if box[0] else None
    if latest is not None:
        ckpt = torch.load(latest, map_location="cpu", weights_only=False)
        assert ckpt["config"] == model_cfg, "Checkpoint config does not match MODEL_D_CONFIG"
        model.load_state_dict(ckpt["model_state"])
        step, tokens_seen = ckpt["step"], ckpt["tokens_seen"]
        same_session = ckpt.get("session_start_epoch") == session_start
        rank_positions = ckpt.get("data_state", {}).get("rank_positions")
        ckpt_phase = ckpt.get("phase_id", "main")
        if ckpt_phase == phase:
            seconds_before = ckpt["seconds_before_session"] if same_session else ckpt["progress_seconds"]
            phase_start_step = ckpt.get("phase_start_step", 0)
        else:
            seconds_before, phase_start_step = 0.0, step
            log(f"[D] new training phase '{phase}' (previous: '{ckpt_phase}') -> fresh LR schedule from step {step:,}")
        log(f"[D] resumed from {latest.name}: step={step:,} tokens={tokens_seen:,} "
            f"phase progress={seconds_before / total_seconds:.1%} ({'same' if same_session else 'new'} session)")
    else:
        log("[D] no checkpoint found -- starting fresh")

    # ---- micro-batch size ---------------------------------------------------
    shapes = cfg["shapes"]
    Pp, Qp = max(((s[0], s[1]) for s in shapes), key=lambda s: s[1])      # longest decoder = most memory
    if cfg.get("fixed_micro_bs") or not cuda:
        micro_bs, sec_micro = cfg.get("fixed_micro_bs") or 2, None
    else:
        res = _probe_micro_batch(model, cfg, device, Pp, Qp)
        fits = [bs for bs, (_, _, ok) in res.items() if ok]
        if not fits:
            model.grad_ckpt = True
            log("[D] nothing fit without activation checkpointing -> turning it on")
            res = _probe_micro_batch(model, cfg, device, Pp, Qp)
            fits = [bs for bs, (_, _, ok) in res.items() if ok]
        for bs, (peak, sec, ok) in res.items():
            log(f"  micro-batch {bs:3d} x 1024 -> " + (f"{peak:5.2f} GB, {sec:.2f}s, {bs * 1024 / sec:,.0f} tok/s/GPU"
                                                      if peak else "OOM") + ("" if ok else "  [no]"))
        best = max(fits) if fits else 0
        if distributed:
            t = torch.tensor([best], device=device)
            dist.all_reduce(t, op=dist.ReduceOp.MIN)
            best = int(t.item())
        assert best > 0, "No micro-batch fits in memory"
        micro_bs, sec_micro = best, res[best][1]
    accum = max(1, round(cfg["tokens_per_step_target"] / (micro_bs * 1024 * world)))
    tokens_per_step = micro_bs * 1024 * world * accum
    log(f"[D] micro-batch {micro_bs} x 1024 tokens x {world} GPU(s) x accum {accum} = {tokens_per_step:,} tokens/step"
        + (f" | probe ~{micro_bs * 1024 * world / sec_micro:,.0f} tok/s" if sec_micro else ""))

    optims = build_optimizers(model, cfg["optimizer"], cfg["peak_lr"], cfg["weight_decay"])
    scaler = make_grad_scaler(cuda)
    if ckpt is not None:
        if ckpt.get("optimizer_kind") == cfg["optimizer"]:
            for o, s in zip(optims, ckpt["optim_states"]):
                o.load_state_dict(s)
            scaler.load_state_dict(ckpt["scaler_state"])
        else:
            log("[D] optimizer changed since the checkpoint -> fresh optimizer state")
        model.grad_ckpt = model.grad_ckpt or ckpt.get("grad_ckpt", False)
        del ckpt
    train_model = model
    if distributed:
        from torch.nn.parallel import DistributedDataParallel as DDP
        train_model = DDP(model, device_ids=[local_rank] if cuda else None, gradient_as_bucket_view=True)

    positions = rank_positions[rank] if rank_positions and rank < len(rank_positions) else None
    files = sorted(Path(cfg["data_dir"]).glob("clean_*.txt"))
    assert files, "No clean_*.txt files in the data folder"
    stream = WindowStream(files, 1024, rank, world, positions)
    data_iter = iter(stream)
    log(f"[D] training files: {[p.name for p in files]} | shapes (prompt, continuation, weight): {shapes}")

    def save_checkpoint(reason):
        gathered = [None] * world
        if distributed:
            dist.all_gather_object(gathered, dict(stream.positions))
        else:
            gathered = [dict(stream.positions)]
        path = None
        if main:
            now = time.time()
            state = {
                "model_state": model.state_dict(), "optim_states": [o.state_dict() for o in optims],
                "optimizer_kind": cfg["optimizer"], "scaler_state": scaler.state_dict(), "step": step,
                "tokens_seen": tokens_seen, "config": model_cfg, "arch_id": ARCH_ID, "run_id": cfg["run_id"],
                "session_start_epoch": session_start, "seconds_before_session": seconds_before,
                "progress_seconds": seconds_before + (now - session_start), "grad_ckpt": model.grad_ckpt,
                "phase_id": phase, "phase_start_step": phase_start_step,
                "data_state": {"rank_positions": gathered, "world": world},
                "train_cfg": {k: cfg[k] for k in ("peak_lr", "shapes", "optimizer", "total_train_hours")},
                "reason": reason,
            }
            path = ckpt_dir / f"{prefix}_step{step}_tokens{tokens_seen}.pt"
            tmp = path.with_name(path.name + ".tmp")
            torch.save(state, tmp)
            tmp.replace(path)
            prune_old_checkpoints(ckpt_dir, prefix, keep=2)
        if distributed:
            dist.barrier()
        return path

    # ---- loop --------------------------------------------------------------
    train_model.train()
    t_loop, last_ckpt, last_log_t, last_log_tokens = time.time(), time.time(), time.time(), tokens_seen
    start_tokens = tokens_seen
    bad_streak, skipped, stop_reason = 0, 0, "unknown"
    ce_log, n_log = torch.zeros((), device=device), 0
    flags = torch.zeros(3, device=device)
    while True:
        if main:
            now = time.time()
            progress = (seconds_before + (now - session_start)) / total_seconds
            stop = 1.0 if (progress >= 1.0 or now >= session_deadline) else 0.0
            do_ckpt = 1.0 if (now - last_ckpt) / 60 >= cfg["checkpoint_every_min"] else 0.0
            flags = torch.tensor([progress, stop, do_ckpt], device=device)
        if distributed:
            dist.broadcast(flags, src=0)
        progress, stop, do_ckpt = float(flags[0]), bool(flags[1] > 0.5), bool(flags[2] > 0.5)
        if stop:
            stop_reason = "schedule complete" if progress >= 1.0 else "session time budget reached"
            break
        lr = cfg["peak_lr"] * lr_factor(step - phase_start_step, progress, cfg["warmup_steps"], cfg["cooldown_frac"], cfg["min_lr_frac"])
        for o in optims:
            for g in o.param_groups:
                g["lr"] = lr
            o.zero_grad(set_to_none=True)
        for micro in range(accum):
            P, Q = choose_shape(shapes, cfg["seed"], step, micro)
            enc, dec_in, tgt = make_example_batch([next(data_iter) for _ in range(micro_bs)], P, Q, device)
            sync_ctx = train_model.no_sync() if (distributed and micro < accum - 1) else contextlib.nullcontext()
            with sync_ctx:
                ce, total = loss_from(train_model(enc, dec_in, targets=tgt), cfg["z_loss_coef"])
                scaler.scale(total / accum).backward()
            ce_log += ce.detach() / accum
            tokens_seen += micro_bs * 1024 * world
        for o in optims:
            scaler.unscale_(o)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"]))
        for o in optims:
            scaler.step(o)
        scaler.update()
        n_log += 1
        step += 1
        if math.isfinite(gnorm):
            bad_streak = 0
        else:
            bad_streak += 1
            skipped += 1
            if bad_streak >= 50:
                stop_reason = "50 non-finite steps in a row"
                break
        if distributed and step % cfg["sync_every"] == 0:
            for p in model.parameters():                     # remove any tiny drift between GPUs
                dist.broadcast(p.data, src=0)
        if step % cfg["log_every"] == 0 or step == 1:
            if distributed:
                dist.all_reduce(ce_log)
                ce_log /= world
            now = time.time()
            tps = (tokens_seen - last_log_tokens) / max(1e-6, now - last_log_t)
            log(f"  step {step:6d}  loss={ce_log.item() / max(1, n_log):.4f}  lr={lr:.2e}  gnorm={gnorm:.2f}  "
                f"tokens={tokens_seen:,}  tok/s={tps:,.0f}  elapsed={(now - t_loop) / 60:.1f} min  "
                f"progress={progress:.1%}  [{datetime.datetime.now():%H:%M:%S}]")
            ce_log.zero_()
            n_log, last_log_t, last_log_tokens = 0, now, tokens_seen
        if do_ckpt:
            path = save_checkpoint("periodic")
            last_ckpt = time.time()
            if main:
                print(f"  [checkpoint saved -> {path.name}]", flush=True)
                if cfg["push"]:
                    push_checkpoint_dataset(cfg["output_dir"], cfg["dataset_id"],
                                            f"{cfg['run_id']}: step={step}, tokens={tokens_seen:,} (in progress)")
            if distributed:
                dist.barrier()
    path = save_checkpoint(stop_reason)
    loop_min = (time.time() - t_loop) / 60
    summary = {"stop_reason": stop_reason, "step": step, "tokens_seen": tokens_seen,
               "session_tokens": tokens_seen - start_tokens, "loop_minutes": round(loop_min, 1),
               "tokens_per_sec": round((tokens_seen - start_tokens) / max(1e-6, loop_min * 60), 1),
               "world": world, "micro_bs": micro_bs, "accum": accum, "tokens_per_step": tokens_per_step,
               "skipped_steps": skipped, "grad_ckpt": model.grad_ckpt, "n_params": n_params, "phase_id": phase,
               "phase_start_step": phase_start_step,
               "final_checkpoint": path.name if path else None, "progress": round(progress, 4),
               "peak_mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if cuda else 0.0}
    if main:
        json.dump(summary, open(Path(cfg["output_dir"]) / "train_summary.json", "w"), indent=1)
        log(f"[D] stopped ({stop_reason}) at step={step:,}, tokens={tokens_seen:,} after {loop_min:.1f} min "
            f"-> {path.name}")
    if distributed:
        dist.barrier()
        dist.destroy_process_group()
    if stop_reason.startswith("50 non-finite"):
        sys.exit(3)


if __name__ == "__main__":
    train_main(sys.argv[1])


Writing /kaggle/working/legal_d.py


In [4]:
%%writefile /kaggle/working/c_model.py
"""c_model.py -- Model C (prefix-LM) classes, copied from p1_07_model_C.ipynb."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

decode = None          # only used by C's text generation, not needed for RR


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x32 = x.float()
        return (x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dtype)


def build_rope_cache(seq_len, d_head, theta=10000.0):
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    freqs = torch.outer(torch.arange(seq_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x [B,H,T,d]; cos/sin [B or 1, T, d/2]. Rotation in fp32."""
    x1, x2 = x[..., 0::2].float(), x[..., 1::2].float()
    cos, sin = cos[:, None].float(), sin[:, None].float()
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2).to(x.dtype)


def prefix_lm_mask(prefix_len, T, device):
    """Bool [B, 1, T, T]: key k visible to query q iff k <= q or k < prefix_len[b]."""
    idx = torch.arange(T, device=device)
    causal = idx[None, :, None] >= idx[None, None, :]
    in_prefix = idx[None, None, :] < prefix_len[:, None, None]
    return (causal | in_prefix)[:, None]


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, h, kv, dh, b = cfg["d_model"], cfg["n_head"], cfg["n_kv_head"], cfg["d_head"], cfg["bias"]
        self.n_head, self.n_kv_head, self.d_head = h, kv, dh
        self.q_proj = nn.Linear(d, h * dh, bias=b)
        self.k_proj = nn.Linear(d, kv * dh, bias=b)
        self.v_proj = nn.Linear(d, kv * dh, bias=b)
        self.o_proj = nn.Linear(h * dh, d, bias=b)
        self.q_norm = RMSNorm(dh) if cfg["qk_norm"] else None
        self.k_norm = RMSNorm(dh) if cfg["qk_norm"] else None

    def forward(self, x, cos, sin, attn_mask, cache=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.d_head).transpose(1, 2)
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        n_rep = self.n_head // self.n_kv_head
        if n_rep > 1:
            k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
        if attn_mask is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, self.n_head * self.d_head))


class GeluMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.up_proj = nn.Linear(cfg["d_model"], cfg["d_ff"], bias=cfg["bias"])
        self.down_proj = nn.Linear(cfg["d_ff"], cfg["d_model"], bias=cfg["bias"])

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg["d_model"])
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg["d_model"])
        self.mlp = GeluMLP(cfg)

    def _inner(self, x, cos, sin, attn_mask, cache):
        x = x + self.attn(self.attn_norm(x), cos, sin, attn_mask, cache)
        return x + self.mlp(self.mlp_norm(x))

    def forward(self, x, cos, sin, attn_mask, cache=None, use_checkpoint=False):
        if use_checkpoint and cache is None and self.training:
            return grad_checkpoint(self._inner, x, cos, sin, attn_mask, None, use_reentrant=False)
        return self._inner(x, cos, sin, attn_mask, cache)


class KVCache:
    def __init__(self, batch_size, n_kv_head, max_len, d_head, device, dtype):
        self.k = torch.zeros(batch_size, n_kv_head, max_len, d_head, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = 0

    def update(self, k, v):
        T = k.shape[2]
        self.k[:, :, self.pos:self.pos + T] = k
        self.v[:, :, self.pos:self.pos + T] = v
        self.pos += T
        return self.k[:, :, :self.pos], self.v[:, :, :self.pos]


class PrefixLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["n_layer"])])
        self.norm_f = RMSNorm(config["d_model"])
        self.head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        cos, sin = build_rope_cache(config["max_seq_len"], config["d_head"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config["n_layer"]))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def encode(self, input_ids, attention_mask=None, mode="prefix", prefix_len=None):
        """Contract method: hidden states. mode 'causal' (prefix 0), 'prefix'
        (needs prefix_len [B]) or 'bidirectional' (whole input is prefix)."""
        B, T = input_ids.shape
        if mode == "causal" or prefix_len is None and mode == "prefix":
            prefix_len = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        elif mode == "bidirectional":
            prefix_len = torch.full((B,), T, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(prefix_len, T, input_ids.device) if bool((prefix_len > 0).any()) else None
        x = self.embed(input_ids)
        cos, sin = self.rope_cos[None, :T], self.rope_sin[None, :T]
        use_ckpt = self.training and USE_GRAD_CHECKPOINTING
        for blk in self.blocks:
            x = blk(x, cos, sin, mask, use_checkpoint=use_ckpt)
        return self.norm_f(x)

    def forward(self, ids, prefix_len=None, targets=None):
        """No targets -> logits. With targets -> [1, 3] = [loss sum, number of
        loss tokens, sum of logsumexp^2 (z-loss)] over positions that predict
        continuation tokens (t >= prefix_len - 1). Sums (not means) so
        DataParallel replicas combine exactly."""
        use_amp = self.training and ids.is_cuda
        if prefix_len is None:
            prefix_len = torch.zeros(ids.shape[0], dtype=torch.long, device=ids.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = self.head(self.encode(ids, mode="prefix", prefix_len=prefix_len))
        if targets is None:
            return logits
        logits32 = logits.float()
        lse = torch.logsumexp(logits32, dim=-1)
        nll = lse - logits32.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pos = torch.arange(ids.shape[1], device=ids.device)[None, :]
        w = (pos >= (prefix_len[:, None] - 1)).float()
        return torch.stack([(nll * w).sum(), w.sum(), (lse.pow(2) * w).sum()]).unsqueeze(0)

    def step(self, ids, positions, caches, attn_mask):
        """Generation step with a KV cache. ids/positions [B, T]; attn_mask
        bool [B, 1, T, cache_len + T]."""
        x = self.embed(ids)
        cos, sin = self.rope_cos[positions], self.rope_sin[positions]
        for blk, cache in zip(self.blocks, caches):
            x = blk(x, cos, sin, attn_mask, cache=cache)
        return self.head(self.norm_f(x))

    def generate(self, input_ids, max_new_tokens, kv_cache=True, sampler="greedy", **kw):
        """Contract method (greedy, one prompt): the prompt is the prefix."""
        assert kv_cache and sampler == "greedy"
        toks = generate_batch(self, [input_ids[0].tolist()], max_new_tokens=max_new_tokens, **kw)[0]
        return torch.tensor([input_ids[0].tolist() + toks], device=input_ids.device)


@torch.no_grad()
def generate_batch(model, prompt_id_lists, max_new_tokens=288, max_context=None, rep_penalty=1.0,
                   no_repeat_ngram=0, target_words=None, banned=(), vocab_limit=None):
    """Greedy continuations for several prompts at once (left-padded). The
    prompt is the bidirectional prefix; continuation is causal. Optional:
    repetition penalty over generated tokens, n-gram blocking inside the
    continuation, stop once every row has > target_words words."""
    model.eval()
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    cfg = model.config
    max_context = max_context or cfg["max_seq_len"]
    prompts = [p[-(max_context - max_new_tokens):] for p in prompt_id_lists]
    B, T0 = len(prompts), max(len(p) for p in prompts)
    pad = torch.tensor([T0 - len(p) for p in prompts], device=device)
    ids = torch.zeros(B, T0, dtype=torch.long, device=device)
    for b, p in enumerate(prompts):
        ids[b, T0 - len(p):] = torch.tensor(p, device=device)
    total = T0 + max_new_tokens
    caches = [KVCache(B, cfg["n_kv_head"], total, cfg["d_head"], device, dtype) for _ in range(cfg["n_layer"])]
    idx = torch.arange(total, device=device)
    key_ok = idx[None, :] >= pad[:, None]
    positions = (idx[None, :] - pad[:, None]).clamp(min=0)
    # Prefill: the prompt attends to itself in both directions (no padding keys);
    # every query may see itself so padded rows are never fully masked.
    mask = (key_ok[:, None, :T0] | torch.eye(T0, dtype=torch.bool, device=device)[None])[:, None]
    logits = model.step(ids, positions[:, :T0], caches, mask)[:, -1, :].float()
    generated = [[] for _ in range(B)]
    seen = [dict() for _ in range(B)]
    for t in range(max_new_tokens):
        if banned:
            logits[:, list(banned)] = -float("inf")
        if vocab_limit is not None:
            logits[:, vocab_limit:] = -float("inf")     # ids the tokenizer cannot decode
        if rep_penalty != 1.0 and t > 0:
            prev = torch.tensor(generated, device=device)
            sc = logits.gather(1, prev)
            logits.scatter_(1, prev, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if no_repeat_ngram and t >= no_repeat_ngram - 1:
            n = no_repeat_ngram
            for b in range(B):
                bad = seen[b].get(tuple(generated[b][-(n - 1):]) if n > 1 else ())
                if bad:
                    logits[b, list(bad)] = -float("inf")
        nxt = logits.argmax(dim=-1)
        for b, tok in enumerate(nxt.tolist()):
            generated[b].append(tok)
            if no_repeat_ngram and len(generated[b]) >= no_repeat_ngram:
                g = generated[b][-no_repeat_ngram:]
                seen[b].setdefault(tuple(g[:-1]), set()).add(g[-1])
        if t == max_new_tokens - 1:
            break
        if target_words and t >= 120 and t % 16 == 0 and all(
                len(decode(g).split()) > target_words for g in generated):
            break
        cur = T0 + t
        logits = model.step(nxt[:, None], positions[:, cur:cur + 1], caches,
                            key_ok[:, None, None, :cur + 1])[:, -1, :].float()
    return generated


def build_model(config):
    """Contract function."""
    return PrefixLM(config)


USE_GRAD_CHECKPOINTING = True




Writing /kaggle/working/c_model.py


In [5]:
%%writefile /kaggle/working/rr_lib.py
"""rr_lib.py -- Phase 3 (Rhetorical Role) library shared by notebooks 1 and 2.

Pipeline (same for every experiment):
  sentences -> Phase-1 backbone (frozen) + adapter -> mean-pooled sentence vector
  -> [+ relative position, domain flag] -> 2-layer BiLSTM over the document's
  sentences (document context) -> linear head -> 13 role logits.

Adapters (own implementations): LoRA, rsLoRA, LoRMA (additive-identity form,
Bihany et al. 2025: W' = ((alpha/r) B A + I) W0), IA3 (Liu et al. 2022),
prompt tuning (Lester et al. 2021), full fine-tuning; knowledge distillation
loss (Hinton et al. 2015).
"""
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

N_LABELS = 13


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
def parse_docs(obj):
    """Accepts the RR json in any of the usual shapes -> list of
    {"id", "text": [sentences], "labels": [..] or None}."""
    if isinstance(obj, dict):
        for k in ("documents", "data", "docs"):
            if k in obj and isinstance(obj[k], list):
                return parse_docs(obj[k])
        docs = []
        for key, v in obj.items():
            if isinstance(v, dict):
                docs.append({"id": str(v.get("id", key)), "text": list(v.get("text") or v.get("sentences") or []),
                             "labels": v.get("labels")})
            elif isinstance(v, list):
                docs.append({"id": str(key), "text": list(v), "labels": None})
        return docs
    docs = []
    for i, v in enumerate(obj):
        docs.append({"id": str(v.get("id", v.get("doc_id", i))), "text": list(v.get("text") or v.get("sentences") or []),
                     "labels": v.get("labels")})
    return docs


def parse_vocab(obj):
    if isinstance(obj, dict):
        for k in ("labels", "label_list", "vocab"):
            if k in obj and isinstance(obj[k], list):
                return [str(x) for x in obj[k]]
        if all(isinstance(v, int) for v in obj.values()):
            return [k for k, _ in sorted(obj.items(), key=lambda kv: kv[1])]
        return [str(k) for k in obj]
    return [str(x) for x in obj]


def windows(n, size, stride):
    """Start/end pairs covering 0..n with overlapping windows."""
    if n <= size:
        return [(0, n)]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return [(s, s + size) for s in starts]


def pad_batch(token_lists, pad_id, device):
    L = max(1, max(len(t) for t in token_lists))
    ids = torch.full((len(token_lists), L), pad_id, dtype=torch.long)
    mask = torch.zeros(len(token_lists), L, dtype=torch.bool)
    for i, t in enumerate(token_lists):
        ids[i, :len(t)] = torch.tensor(t, dtype=torch.long)
        mask[i, :len(t)] = True
    return ids.to(device), mask.to(device)


# ---------------------------------------------------------------------------
# Adapters
# ---------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W0 x + s * B A x  (s = alpha / r; rsLoRA: alpha / sqrt(r))."""

    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / (math.sqrt(r) if rs else r)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """LoRMA (additive-identity form): y = ((s B A) + I) W0 x = h + s B A h, h = W0 x.
    B = 0 at start, so the adapted layer starts exactly as the Phase-1 layer."""

    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + F.linear(F.linear(h, self.A), self.B) * self.scale


class IA3Out(nn.Module):
    """Learned per-channel scaling of a layer's output (keys, values)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class IA3In(nn.Module):
    """Learned per-channel scaling of a layer's input (the MLP activation before down_proj)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


LINEAR_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


def _replace(module, name, new):
    parent = module
    parts = name.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], new)


def apply_adapter(backbone, kind, r=16, alpha=32):
    """Freeze the backbone and insert the adapter. Returns the number of
    adapter parameters."""
    for p in backbone.parameters():
        p.requires_grad_(False)
    if kind == "full":
        for p in backbone.parameters():
            p.requires_grad_(True)
        return sum(p.numel() for p in backbone.parameters())
    if kind == "prompt":
        return 0                                         # soft prompt lives in the Backbone wrapper
    names = [n for n, m in backbone.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in LINEAR_TARGETS]
    for n in names:
        lin = backbone.get_submodule(n)
        leaf = n.split(".")[-1]
        if kind == "lora":
            _replace(backbone, n, LoRALinear(lin, r, alpha))
        elif kind == "rslora":
            _replace(backbone, n, LoRALinear(lin, r, alpha, rs=True))
        elif kind == "lorma":
            _replace(backbone, n, LoRMALinear(lin, r, alpha))
        elif kind == "ia3":
            if leaf in ("k_proj", "v_proj"):
                _replace(backbone, n, IA3Out(lin))
            elif leaf == "down_proj":
                _replace(backbone, n, IA3In(lin))
        else:
            raise ValueError(kind)
    return sum(p.numel() for p in backbone.parameters() if p.requires_grad)


# ---------------------------------------------------------------------------
# Backbones: D encoder, C (both ways), A / B (left to right, sentence fed twice)
# ---------------------------------------------------------------------------
class Backbone(nn.Module):
    """sentence token ids [S, L] (right-padded) -> sentence vectors [S, d]."""

    def __init__(self, kind, model, d_model, n_prompt=0):
        super().__init__()
        self.kind, self.m, self.d_model = kind, model, d_model
        self.prompt = nn.Parameter(torch.randn(n_prompt, d_model) * 0.02) if n_prompt else None

    def forward(self, ids, mask):
        amp = torch.autocast(device_type="cuda", dtype=torch.float16, enabled=ids.is_cuda)
        with amp:
            if self.kind == "D":
                h = self.m.encode(ids, mask)
            elif self.kind == "C":
                h = self.m.encode(ids, mode="prefix", prefix_len=mask.sum(1))
            else:
                h, mask = self._causal_twice(ids, mask)
        h = h.float()
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1.0)

    def _causal_twice(self, ids, mask):
        """A/B read left to right: feed [sentence, sentence] and pool the second
        copy, so every pooled token has seen the whole sentence."""
        lens = mask.sum(1)
        S, L = ids.shape
        two = torch.zeros(S, 2 * L, dtype=ids.dtype, device=ids.device)
        second = torch.zeros(S, 2 * L, dtype=torch.bool, device=ids.device)
        for i in range(S):
            n = int(lens[i])
            two[i, :n] = ids[i, :n]
            two[i, n:2 * n] = ids[i, :n]
            second[i, n:2 * n] = True
        return self.m.hidden_states(two), second


class RRModel(nn.Module):
    def __init__(self, backbone, d_model, use_context=True, hidden=256, dropout=0.1):
        super().__init__()
        self.backbone, self.use_context = backbone, use_context
        self.proj = nn.Sequential(nn.Linear(d_model + 3, 2 * hidden), nn.GELU(), nn.Dropout(dropout))
        self.lstm = nn.LSTM(2 * hidden, hidden, num_layers=2, batch_first=True, bidirectional=True,
                            dropout=dropout) if use_context else None
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(2 * hidden, N_LABELS)

    def sentence_vectors(self, token_lists, pad_id, device, chunk=64):
        vecs = []
        for s in range(0, len(token_lists), chunk):
            ids, mask = pad_batch(token_lists[s:s + chunk], pad_id, device)
            vecs.append(self.backbone(ids, mask))
        return torch.cat(vecs, 0)

    def forward(self, token_lists, positions, domain, pad_id, device):
        v = self.sentence_vectors(token_lists, pad_id, device)                 # [S, d]
        pos = torch.tensor(positions, dtype=torch.float32, device=device)[:, None]
        dom = torch.zeros(len(token_lists), 2, device=device)
        dom[:, domain] = 1.0
        x = self.proj(torch.cat([v, pos, dom], -1))
        if self.lstm is not None:
            x = self.lstm(x[None])[0][0]
        return self.out(self.drop(x))                                         # [S, 13]


# ---------------------------------------------------------------------------
# Metrics and per-class bias
# ---------------------------------------------------------------------------
def macro_f1(y_true, y_pred):
    """Macro-F1 over the labels present in gold or prediction (sklearn's default)."""
    t = np.asarray(y_true, dtype=int)
    p = np.asarray(y_pred, dtype=int)
    if t.size == 0:
        return 0.0
    k = max(t.max(), p.max()) + 1
    tp = np.bincount(t[t == p], minlength=k).astype(float)
    n_true = np.bincount(t, minlength=k).astype(float)
    n_pred = np.bincount(p, minlength=k).astype(float)
    present = (n_true + n_pred) > 0
    denom = n_true + n_pred
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return float(f1[present].mean())


def per_class(y_true, y_pred, names):
    rows = []
    for c, name in enumerate(names):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        rows.append({"label": name, "support": tp + fn, "precision": round(prec, 4), "recall": round(rec, 4),
                     "f1": round(2 * prec * rec / (prec + rec), 4) if prec + rec else 0.0})
    return rows


def fit_bias(logits, y, rounds=3, grid=np.linspace(-3, 3, 25)):
    """One additive bias per class (on log-probabilities), chosen by coordinate
    ascent to maximise macro-F1 on the given sentences."""
    lp = torch.log_softmax(torch.as_tensor(logits, dtype=torch.float32), -1).numpy()
    b = np.zeros(lp.shape[1])
    best = macro_f1(list(y), list((lp + b).argmax(1)))
    for _ in range(rounds):
        for c in range(lp.shape[1]):
            keep = b[c]
            for v in grid:
                b[c] = v
                f = macro_f1(list(y), list((lp + b).argmax(1)))
                if f > best + 1e-9:
                    best, keep = f, v
            b[c] = keep
    return b, best


def apply_bias(logits, b):
    return (torch.log_softmax(torch.as_tensor(logits, dtype=torch.float32), -1).numpy() + b).argmax(1)


# ---------------------------------------------------------------------------
# Training / inference
# ---------------------------------------------------------------------------
@torch.no_grad()
def predict_docs(model, docs, pad_id, device, win, stride):
    """Raw logits per document [n_sent, 13]; overlapping windows averaged."""
    model.eval()
    out = {}
    for d in docs:
        n = len(d["ids"])
        acc = torch.zeros(n, N_LABELS)
        cnt = torch.zeros(n, 1)
        for s, e in windows(n, win, stride):
            pos = [i / max(1, n - 1) for i in range(s, e)]
            lg = model(d["ids"][s:e], pos, d["domain"], pad_id, device).float().cpu()
            acc[s:e] += lg
            cnt[s:e] += 1
        out[d["key"]] = acc / cnt
    return out


def evaluate(logits_by_key, docs):
    y = [l for d in docs for l in d["labels"]]
    p = [int(i) for d in docs for i in logits_by_key[d["key"]].argmax(1)]
    return macro_f1(y, p)


def train_experiment(exp, data, make_backbone, device, log=print, teacher=None):
    """exp: dict(name, model, adapter, lr, lr_head, r, alpha, use_context, n_prompt,
    epochs, patience, budget_min, win, stride, kd_alpha, kd_T, seed).
    teacher: {doc_key: logits tensor} for knowledge distillation (or None)."""
    random.seed(exp["seed"])
    torch.manual_seed(exp["seed"])
    backbone_model, d_model, pad_id = make_backbone(exp["model"])
    apply_adapter(backbone_model, exp["adapter"], exp.get("r", 16), exp.get("alpha", 32))
    backbone = Backbone(exp["model"], backbone_model, d_model, exp.get("n_prompt", 0))
    model = RRModel(backbone, d_model, use_context=exp.get("use_context", True)).to(device)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.") and p.requires_grad]
    adapter_params = [p for n, p in model.named_parameters() if n.startswith("backbone.") and p.requires_grad]
    n_trainable = sum(p.numel() for p in head_params + adapter_params)
    n_backbone = sum(p.numel() for p in backbone_model.parameters())
    log(f"[{exp['name']}] backbone {exp['model']} {n_backbone / 1e6:.1f}M params | adapter '{exp['adapter']}' "
        f"{sum(p.numel() for p in adapter_params):,} | head+context {sum(p.numel() for p in head_params):,} | "
        f"trainable {n_trainable:,} ({100 * n_trainable / (n_backbone + sum(p.numel() for p in head_params)):.2f}%)")
    groups = [{"params": head_params, "lr": exp["lr_head"]}]
    if adapter_params:
        groups.append({"params": adapter_params, "lr": exp["lr"]})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    train, dev = data["train"], data["dev"]
    counts = np.bincount([l for d in train for l in d["labels"]], minlength=N_LABELS).astype(float)
    w = 1.0 / np.sqrt(np.maximum(counts, 1.0))
    w = w / (w * counts).sum() * counts.sum()                     # mean weight per sentence = 1
    class_w = torch.tensor(w, dtype=torch.float32, device=device)
    win, stride = exp["win"], exp["stride"]
    steps_per_epoch = sum(len(windows(len(d["ids"]), win, stride)) for d in train)
    total = steps_per_epoch * exp["epochs"]
    warm = max(10, total // 20)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * max(0.1, 1 - max(0, s - warm) / max(1, total - warm)))
    log(f"[{exp['name']}] {len(train)} train docs, {steps_per_epoch} windows/epoch, up to {exp['epochs']} epochs, "
        f"budget {exp['budget_min']} min")

    t0 = time.time()
    best_f1, best_state, bad, step, history = -1.0, None, 0, 0, []
    trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
    for epoch in range(exp["epochs"]):
        model.train()
        items = [(d, s, e) for d in train for s, e in windows(len(d["ids"]), win, stride)]
        random.shuffle(items)
        run_loss = 0.0
        for d, s, e in items:
            n = len(d["ids"])
            pos = [i / max(1, n - 1) for i in range(s, e)]
            y = torch.tensor(d["labels"][s:e], device=device)
            logits = model(d["ids"][s:e], pos, d["domain"], pad_id, device).float()
            loss = F.cross_entropy(logits, y, weight=class_w)
            if teacher is not None:
                T = exp["kd_T"]
                t_logits = teacher[d["key"]][s:e].to(device).float()
                kd = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(t_logits / T, -1), reduction="batchmean") * T * T
                loss = exp["kd_alpha"] * kd + (1 - exp["kd_alpha"]) * loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            run_loss += float(loss.detach())
            if (time.time() - t0) / 60 > exp["budget_min"]:
                break
        dev_logits = predict_docs(model, dev, pad_id, device, win, stride)
        f1 = evaluate(dev_logits, dev)
        history.append({"epoch": epoch + 1, "train_loss": round(run_loss / max(1, len(items)), 4),
                        "dev_macro_f1": round(f1, 4), "minutes": round((time.time() - t0) / 60, 1)})
        log(f"[{exp['name']}] epoch {epoch + 1}: train loss {history[-1]['train_loss']:.4f} | dev macro-F1 {f1:.4f} "
            f"| {history[-1]['minutes']} min")
        if f1 > best_f1:
            best_f1, bad = f1, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.state_dict().items() if n in trainable_names}
        else:
            bad += 1
        if bad >= exp["patience"] or (time.time() - t0) / 60 > exp["budget_min"]:
            break
    model.load_state_dict(best_state, strict=False)
    info = {"n_backbone": n_backbone, "n_trainable": n_trainable, "train_minutes": round((time.time() - t0) / 60, 1),
            "history": history, "best_dev_macro_f1_raw": round(best_f1, 4),
            "peak_mem_gb": round(torch.cuda.max_memory_allocated(device) / 1e9, 2) if device.type == "cuda" else 0.0}
    return model, pad_id, info, best_state


def run_and_save(exp, data, make_backbone, device, out_dir, vocab, log=print, teacher=None, save_train_logits=False):
    """Train, then write logits, metrics, bias and submission for one experiment."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model, pad_id, info, state = train_experiment(exp, data, make_backbone, device, log, teacher)
    win, stride = exp["win"], exp["stride"]
    t0 = time.time()
    dev_logits = predict_docs(model, data["dev"], pad_id, device, win, stride)
    n_dev_sent = sum(len(d["ids"]) for d in data["dev"])
    info["inference_sent_per_s"] = round(n_dev_sent / max(1e-6, time.time() - t0), 1)
    test_logits = predict_docs(model, data["test"], pad_id, device, win, stride)
    splits = {"dev": dev_logits, "test": test_logits}
    if save_train_logits:
        splits["train"] = predict_docs(model, data["train"], pad_id, device, win, stride)
    torch.save(splits, out_dir / "logits.pt")
    torch.save(state, out_dir / "trainable_state.pt")

    dev = data["dev"]
    y = np.array([l for d in dev for l in d["labels"]])
    L = torch.cat([dev_logits[d["key"]] for d in dev]).numpy()
    raw_pred = L.argmax(1)
    info["dev_macro_f1_raw"] = round(macro_f1(list(y), list(raw_pred)), 4)
    # honest estimate of the bias step: fit on half the dev documents, score the other half
    half = len(dev) // 2
    mask_a = np.zeros(len(y), bool)
    mask_a[:sum(len(d["ids"]) for d in dev[:half])] = True          # first half of the dev documents
    pred_cv = raw_pred.copy()
    if half and mask_a.any() and (~mask_a).any():
        b_a, _ = fit_bias(L[mask_a], y[mask_a])
        b_b, _ = fit_bias(L[~mask_a], y[~mask_a])
        pred_cv[~mask_a] = apply_bias(L[~mask_a], b_a)
        pred_cv[mask_a] = apply_bias(L[mask_a], b_b)
    info["dev_macro_f1_bias_2fold"] = round(macro_f1(list(y), list(pred_cv)), 4)
    bias, fitted = fit_bias(L, y)
    info["dev_macro_f1_bias_fitted_on_dev"] = round(fitted, 4)
    use_bias = info["dev_macro_f1_bias_2fold"] > info["dev_macro_f1_raw"]
    info["bias_used_for_test"] = bool(use_bias)
    final_dev_pred = apply_bias(L, bias) if use_bias else raw_pred
    for dom, name in ((0, "CL"), (1, "IT")):
        dd = [d for d in dev if d["domain"] == dom]
        if dd:
            yy = [l for d in dd for l in d["labels"]]
            Ld = torch.cat([dev_logits[d["key"]] for d in dd]).numpy()
            pp = apply_bias(Ld, bias) if use_bias else Ld.argmax(1)
            info[f"dev_macro_f1_{name}"] = round(macro_f1(yy, list(pp)), 4)
    info["per_class_dev"] = per_class(list(y), list(final_dev_pred), vocab)
    conf = np.zeros((N_LABELS, N_LABELS), int)
    for t, p in zip(y, final_dev_pred):
        conf[t, p] += 1
    np.savetxt(out_dir / "dev_confusion.csv", conf, fmt="%d", delimiter=",", header=",".join(vocab))

    preds = {}
    for d in data["test"]:
        lg = test_logits[d["key"]].numpy()
        p = apply_bias(lg, bias) if use_bias else lg.argmax(1)
        labels = [vocab[int(i)] for i in p]
        assert len(labels) == len(d["text"]), d["id"]
        assert d["id"] not in preds, f"duplicate test id {d['id']}"
        preds[d["id"]] = labels
    with open(out_dir / "prediction.json", "w") as f:
        json.dump(preds, f)
    import zipfile
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")
    info["exp"] = exp
    info["bias"] = [round(float(x), 3) for x in bias]
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(info, f, indent=1)
    log(f"[{exp['name']}] DONE dev macro-F1 raw {info['dev_macro_f1_raw']:.4f} | with class bias (2-fold honest) "
        f"{info['dev_macro_f1_bias_2fold']:.4f} | CL {info.get('dev_macro_f1_CL', float('nan')):.4f} "
        f"IT {info.get('dev_macro_f1_IT', float('nan')):.4f} | -> {out_dir / 'submission.zip'}")
    return info


def self_test(make_tiny, device):
    """Adapters start as the identity (output unchanged); sentence vectors do not
    depend on padding; the full pipeline runs forward and backward."""
    torch.manual_seed(0)
    ids_a = [5, 9, 7, 11, 6, 8]
    ids_b = [4, 10, 12]
    for kind in ("D", "C"):
        for adapter in ("lora", "rslora", "lorma", "ia3"):
            m, d, pad = make_tiny(kind)
            m = m.to(device)
            bb = Backbone(kind, m, d).to(device)
            ids, mask = pad_batch([ids_a, ids_b], pad, device)
            with torch.no_grad():
                before = bb(ids, mask)
            apply_adapter(m, adapter)
            m.to(device)
            with torch.no_grad():
                after = bb(ids, mask)
                alone, mask1 = pad_batch([ids_b], pad, device)
                single = bb(alone, mask1)
            assert (before - after).abs().max() < 1e-4, f"{kind}/{adapter} does not start as the identity"
            assert (single[0] - after[1]).abs().max() < 1e-4, f"{kind}: padding changes the sentence vector"
        rr = RRModel(bb, d).to(device)
        logits = rr([ids_a, ids_b, ids_a], [0.0, 0.5, 1.0], 1, pad, device)
        F.cross_entropy(logits, torch.tensor([0, 3, 12], device=device)).backward()
        assert logits.shape == (3, N_LABELS)
        assert any(p.grad is not None for n, p in rr.named_parameters() if n.startswith("backbone.") and p.requires_grad)
    print("rr_lib self-test passed: adapters start as identity (LoRA, rsLoRA, LoRMA, IA3) for D and C, "
          "padding-safe sentence vectors, forward/backward ok.")


Writing /kaggle/working/rr_lib.py


In [6]:
%%writefile /kaggle/working/rr_train.py
"""rr_train.py -- run ONE Phase-3 experiment: python rr_train.py <config.json> <experiment> <gpu index or -1>"""
import json
import sys
from pathlib import Path

import torch

cfg = json.load(open(sys.argv[1]))
name, gpu = sys.argv[2], int(sys.argv[3])
sys.path.insert(0, cfg["work_dir"])
import c_model  # noqa: E402
import legal_d  # noqa: E402
import rr_lib as R  # noqa: E402

device = torch.device(f"cuda:{gpu}") if gpu >= 0 and torch.cuda.is_available() else torch.device("cpu")
if device.type == "cuda":
    torch.cuda.set_device(device)
legal_d.load_tokenizer(cfg["tokenizer_path"])
data = torch.load(cfg["data_path"], weights_only=False)


def make_backbone(kind):
    if kind == "D":
        ck = torch.load(cfg["d_ckpt"], map_location="cpu", weights_only=False)
        m = legal_d.build_model(ck["config"])
        m.load_state_dict(ck["model_state"])
        m.dec_blocks = torch.nn.ModuleList()          # encoder only: the decoder is not used for RR
        m.grad_ckpt = True
        return m, ck["config"]["d_model"], data["pad_id"]
    if kind == "C":
        ck = torch.load(cfg["c_ckpt"], map_location="cpu", weights_only=False)
        m = c_model.build_model(ck["config"])
        m.load_state_dict(ck["model_state"])
        c_model.USE_GRAD_CHECKPOINTING = True
        return m, ck["config"]["d_model"], data["pad_id"]
    raise ValueError(kind)


def log(*a):
    print(*a, flush=True)


exp = dict(cfg["common"], **cfg["experiments"][name], name=name)
R.run_and_save(exp, data, make_backbone, device, Path(cfg["out_dir"]) / name, data["vocab"], log=log,
               save_train_logits=exp.get("save_train_logits", False))


Writing /kaggle/working/rr_train.py


## 3. Data: download, check, tokenize

Prints documents / sentences / label counts per domain and split, sentence lengths, and checks that CL
and IT test documents do not share IDs (one prediction file holds both).


In [7]:
sys.path.insert(0, str(WORK_DIR))
import c_model  # noqa: E402
import legal_d  # noqa: E402
import rr_lib as R  # noqa: E402
from huggingface_hub import hf_hub_download  # noqa: E402
from kaggle_secrets import UserSecretsClient  # noqa: E402

HF_TOKEN = UserSecretsClient().get_secret("HF")
REPO = "Exploration-Lab/CS-781-Capstone"
MAX_TOK = 128           # sentence length cap in T2 tokens


def hf_json(path):
    return json.load(open(hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)))


legal_d.load_tokenizer(TOKENIZER_PATH)
sp = legal_d.sp
PAD_ID = sp.pad_id() if sp.pad_id() >= 0 else 0
vocab = R.parse_vocab(hf_json("Phase_3_RR/Train/label_vocab.json"))
print("Labels:", vocab)
assert len(vocab) == R.N_LABELS, f"expected 13 labels, got {len(vocab)}"
FILES = {("CL", "train"): "Phase_3_RR/Train/CL_train.json", ("CL", "dev"): "Phase_3_RR/Train/CL_dev.json",
         ("IT", "train"): "Phase_3_RR/Train/IT_train.json", ("IT", "dev"): "Phase_3_RR/Train/IT_dev.json",
         ("CL", "test"): "Phase_3_RR/Test/CL_test.json", ("IT", "test"): "Phase_3_RR/Test/IT_test.json"}
data = {"train": [], "dev": [], "test": [], "vocab": vocab, "pad_id": PAD_ID}
lens, unknown = [], {}
for (dom, split), path in FILES.items():
    docs = R.parse_docs(hf_json(path))
    for d in docs:
        toks = [(sp.encode(s, out_type=int)[:MAX_TOK] or [legal_d.EOS_ID]) for s in d["text"]]
        lens += [len(t) for t in toks]
        labels = None
        if split != "test":
            assert d["labels"] is not None and len(d["labels"]) == len(d["text"]), f"{dom} {split} {d['id']}: labels/text mismatch"
            for l in d["labels"]:
                if l not in vocab:
                    unknown[l] = unknown.get(l, 0) + 1
            labels = [vocab.index(l) if l in vocab else vocab.index("None") for l in d["labels"]]
        data[split].append({"key": f"{dom}/{d['id']}", "id": d["id"], "domain": 0 if dom == "CL" else 1,
                            "text": d["text"], "ids": toks, "labels": labels})
assert not unknown, f"labels not in label_vocab.json: {unknown}"

for split in ("train", "dev", "test"):
    for dom, name in ((0, "CL"), (1, "IT")):
        dd = [d for d in data[split] if d["domain"] == dom]
        ns = [len(d["text"]) for d in dd]
        print(f"{split:5s} {name}: {len(dd):4d} docs, {sum(ns):6d} sentences (per doc: min {min(ns) if ns else 0}, "
              f"median {sorted(ns)[len(ns) // 2] if ns else 0}, max {max(ns) if ns else 0})")
lens = sorted(lens)
print(f"Sentence length (T2 tokens, capped {MAX_TOK}): median {lens[len(lens) // 2]}, 95th pct {lens[int(.95 * len(lens))]}, "
      f"share hitting the cap {sum(l >= MAX_TOK for l in lens) / len(lens):.1%}")
for split in ("train", "dev"):
    cnt = [0] * len(vocab)
    for d in data[split]:
        for l in d["labels"]:
            cnt[l] += 1
    print(f"{split} label counts:", {vocab[i]: cnt[i] for i in sorted(range(len(vocab)), key=lambda i: -cnt[i])})
test_ids = [d["id"] for d in data["test"]]
assert len(test_ids) == len(set(test_ids)), "CL and IT test documents share IDs -- one prediction file cannot hold both!"
print(f"Test: {len(test_ids)} documents, IDs unique across CL and IT.")
DATA_PATH = OUT_DIR / "rr_data.pt"
torch.save(data, DATA_PATH)


label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

train CL:   40 docs,  10593 sentences (per doc: min 19, median 123, max 1816)
train IT:   40 docs,   6252 sentences (per doc: min 25, median 129, max 442)
dev   CL:    5 docs,   1396 sentences (per doc: min 32, median 184, max 848)
dev   IT:    5 docs,    746 sentences (per doc: min 45, median 120, max 382)
test  CL:    5 docs,   1339 sentences (per doc: min 18, median 114, max 942)
test  IT:    5 docs,    858 sentences (per doc: min 40, median 159, max 420)
Sentence length (T2 tokens, capped 128): median 33, 95th pct 91, share hitting the cap 1.9%
train label counts: {'RatioOfTheDecision': 4301, 'Fact': 4098, 'PrecedentReliedUpon': 1945, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'RulingByLowerCourt': 874, 'PrecedentNotReliedUpon': 569, 'RulingByPresentCourt': 307, 'None': 297, 'Dissent': 284, 'Issue': 107, 'PrecedentOverruled': 3}
dev label counts: {'Fact': 444, 'RatioOfTheDecision': 403, 'ArgumentPetitioner': 400, 'ArgumentRespondent': 272, 'PrecedentRe

## 4. Self-test (tiny models, a few seconds)

In [8]:
def make_tiny(kind):
    if kind == "D":
        cfg = dict(legal_d.MODEL_D_CONFIG, vocab_size=64, d_model=32, n_enc_layer=2, n_dec_layer=1, n_head=4,
                   n_kv_head=2, d_head=8, d_ff=48, max_enc_len=64, max_dec_len=64)
        return legal_d.build_model(cfg).eval(), 32, 0
    cfg = dict(vocab_size=64, d_model=32, n_layer=2, n_head=4, n_kv_head=2, d_head=8, d_ff=48, max_seq_len=64,
               qk_norm=True, bias=True, mlp="gelu", tokenizer_id="T2", attn_type="prefix", tied_embeddings=True)
    return c_model.build_model(cfg).eval(), 32, 0


R.self_test(make_tiny, torch.device("cpu"))


rr_lib self-test passed: adapters start as identity (LoRA, rsLoRA, LoRMA, IA3) for D and C, padding-safe sentence vectors, forward/backward ok.


## 5. Experiment settings

Shared settings in `COMMON`; each experiment changes only its technique (or, for the ablation, removes
the context layer). Windows of 64 sentences with stride 48 cover long documents; overlapping
predictions are averaged.


In [9]:
COMMON = dict(epochs=10, patience=3, budget_min=35, win=64, stride=48, seed=13, r=16, alpha=32,
              lr_head=1e-3, use_context=True)
EXPERIMENTS = {
    "E1": dict(model="D", adapter="lorma", lr=5e-4, save_train_logits=True),
    "E2": dict(model="D", adapter="lora", lr=5e-4, save_train_logits=True),
    "E3": dict(model="C", adapter="ia3", lr=3e-3),
    "E1_noctx": dict(model="D", adapter="lorma", lr=5e-4, use_context=False),
}
RUN_ORDER = ["E1", "E2", "E3", "E1_noctx"]       # two at a time on two GPUs
CFG = {"work_dir": str(WORK_DIR), "tokenizer_path": str(TOKENIZER_PATH), "data_path": str(DATA_PATH),
       "d_ckpt": str(D_CKPT), "c_ckpt": str(C_CKPT), "out_dir": str(OUT_DIR), "common": COMMON,
       "experiments": EXPERIMENTS}
CONFIG_PATH = WORK_DIR / "rr_config_nb1.json"
json.dump(CFG, open(CONFIG_PATH, "w"), indent=1)
print(json.dumps(EXPERIMENTS, indent=1))


{
 "E1": {
  "model": "D",
  "adapter": "lorma",
  "lr": 0.0005,
  "save_train_logits": true
 },
 "E2": {
  "model": "D",
  "adapter": "lora",
  "lr": 0.0005,
  "save_train_logits": true
 },
 "E3": {
  "model": "C",
  "adapter": "ia3",
  "lr": 0.003
 },
 "E1_noctx": {
  "model": "D",
  "adapter": "lorma",
  "lr": 0.0005,
  "use_context": false
 }
}


## 6. Run (two experiments at a time)

In [10]:
def stream(proc, tag):
    for line in proc.stdout:
        print(line if line.startswith(f"[{tag}]") else f"[{tag}] {line}", end="", flush=True)


queue, running, codes = list(RUN_ORDER), {}, {}
slots = list(range(N_GPUS))[:2] if N_GPUS else [-1]
t_all = time.time()
while queue or running:
    for slot in slots:
        if slot not in running and queue:
            name = queue.pop(0)
            env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
            proc = subprocess.Popen([sys.executable, str(WORK_DIR / "rr_train.py"), str(CONFIG_PATH), name, str(slot)],
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
            th = threading.Thread(target=stream, args=(proc, name), daemon=True)
            th.start()
            running[slot] = (name, proc, th)
            print(f">>> started {name} on {'GPU ' + str(slot) if slot >= 0 else 'CPU'}", flush=True)
    for slot, (name, proc, th) in list(running.items()):
        if proc.poll() is not None:
            th.join()
            codes[name] = proc.returncode
            print(f">>> {name} finished with code {proc.returncode} ({(time.time() - t_all) / 60:.1f} min so far)", flush=True)
            del running[slot]
    time.sleep(5)
print("Exit codes:", codes)


>>> started E1 on GPU 0
>>> started E2 on GPU 1
[E1] backbone D 102.7M params | adapter 'lorma' 2,654,208 | head+context 3,555,853 | trainable 6,210,061 (5.84%)
[E2] backbone D 102.7M params | adapter 'lora' 2,605,056 | head+context 3,555,853 | trainable 6,160,909 (5.80%)
[E2] 80 train docs, 365 windows/epoch, up to 10 epochs, budget 35 min
[E1] 80 train docs, 365 windows/epoch, up to 10 epochs, budget 35 min
[E1] epoch 1: train loss 2.2465 | dev macro-F1 0.2208 | 3.7 min
[E2] epoch 1: train loss 2.1406 | dev macro-F1 0.2687 | 4.0 min
[E1] epoch 2: train loss 1.7085 | dev macro-F1 0.3989 | 7.5 min
[E2] epoch 2: train loss 1.5970 | dev macro-F1 0.4182 | 8.1 min
[E1] epoch 3: train loss 1.3776 | dev macro-F1 0.3983 | 11.2 min
[E2] epoch 3: train loss 1.2481 | dev macro-F1 0.4206 | 12.2 min
[E1] epoch 4: train loss 1.1377 | dev macro-F1 0.4863 | 15.0 min
[E2] epoch 4: train loss 0.9706 | dev macro-F1 0.5055 | 16.2 min
[E1] epoch 5: train loss 0.9311 | dev macro-F1 0.5364 | 18.8 min
[E2] e

## 7. Results, teacher choice, submissions

In [11]:
rows = []
for name in RUN_ORDER:
    mpath = OUT_DIR / name / "metrics.json"
    if not mpath.exists():
        print(f"{name}: no metrics (see its log above)")
        continue
    m = json.load(open(mpath))
    rows.append({"exp": name, "model": m["exp"]["model"], "technique": m["exp"]["adapter"],
                 "context": m["exp"]["use_context"], "trainable_params": m["n_trainable"],
                 "backbone_params": m["n_backbone"], "dev_f1_raw": m["dev_macro_f1_raw"],
                 "dev_f1_bias_2fold": m["dev_macro_f1_bias_2fold"], "bias_used": m["bias_used_for_test"],
                 "dev_f1_CL": m.get("dev_macro_f1_CL"), "dev_f1_IT": m.get("dev_macro_f1_IT"),
                 "train_min": m["train_minutes"], "peak_mem_gb": m["peak_mem_gb"],
                 "infer_sent_per_s": m["inference_sent_per_s"], "epochs_run": len(m["history"])})
with open(OUT_DIR / "runs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)
print(f"{'exp':9s} {'model':5s} {'technique':9s} {'ctx':5s} {'trainable':>10s} {'raw F1':>7s} {'bias F1':>7s} "
      f"{'CL':>6s} {'IT':>6s} {'min':>5s}")
for r in rows:
    print(f"{r['exp']:9s} {r['model']:5s} {r['technique']:9s} {str(r['context']):5s} {r['trainable_params']:>10,} "
          f"{r['dev_f1_raw']:7.4f} {r['dev_f1_bias_2fold']:7.4f} {r['dev_f1_CL'] or 0:6.4f} {r['dev_f1_IT'] or 0:6.4f} "
          f"{r['train_min']:5.1f}")

teachers = [r for r in rows if r["exp"] in ("E1", "E2")]
if teachers:
    t = max(teachers, key=lambda r: r["dev_f1_raw"])
    json.dump({"teacher": t["exp"], "dev_f1_raw": t["dev_f1_raw"], "d_ckpt": D_CKPT.name},
              open(OUT_DIR / "teacher.json", "w"), indent=1)
    print(f"\nDistillation teacher for notebook 2: {t['exp']} (dev macro-F1 {t['dev_f1_raw']:.4f})")
abl = {r["exp"]: r for r in rows}
if "E1" in abl and "E1_noctx" in abl:
    print(f"Context ablation (dev, raw): with context {abl['E1']['dev_f1_raw']:.4f} vs sentence-only "
          f"{abl['E1_noctx']['dev_f1_raw']:.4f}")
print("\nSubmit these (Output tab):")
for name in ("E1", "E2", "E3"):
    z = OUT_DIR / name / "submission.zip"
    print(f"  {name}: {z}" if z.exists() else f"  {name}: MISSING")


exp       model technique ctx    trainable  raw F1 bias F1     CL     IT   min
E1        D     lorma     True   6,210,061  0.5749  0.5351 0.5386 0.5168  35.2
E2        D     lora      True   6,160,909  0.5159  0.4551 0.4367 0.5126  32.5
E3        C     ia3       True   3,606,029  0.5473  0.4838 0.5014 0.5013  32.3
E1_noctx  D     lorma     False  3,056,141  0.4109  0.3842 0.3287 0.4637  29.8

Distillation teacher for notebook 2: E1 (dev macro-F1 0.5749)
Context ablation (dev, raw): with context 0.5749 vs sentence-only 0.4109

Submit these (Output tab):
  E1: /kaggle/working/p3_nb1/E1/submission.zip
  E2: /kaggle/working/p3_nb1/E2/submission.zip
  E3: /kaggle/working/p3_nb1/E3/submission.zip
